Question1: Easy (CO2) – Dataset1: Normalize a vector using CUDA. (Dataset1: Vector Operations Dataset (10 Million Floating-Point Values))

In [1]:
# 1. Install & import everything

!pip install numba -q

import numpy as np
import math
import numba          # needed for numba.float32 inside kernels
from numba import cuda
import time

In [2]:
# 2. Create the dataset (10M floats)

N = 10_000_000
np.random.seed(42)
host_vector = np.random.rand(N).astype(np.float32)

print(f"Dataset size: {N} elements, dtype: {host_vector.dtype}")
print("First 5 original values:", host_vector[:5])

Dataset size: 10000000 elements, dtype: float32
First 5 original values: [0.37454012 0.9507143  0.7319939  0.5986585  0.15601864]


In [3]:
# 3. Define CUDA kernels

@cuda.jit
def square_reduce_kernel(data, partial_sums, n):
    """
    Each block computes the sum of squares of its own chunk.
    The result is stored in partial_sums[blockIdx.x].
    """
    # Shared memory for reduction inside the block
    sdata = cuda.shared.array(256, dtype=numba.float32)

    tid = cuda.threadIdx.x
    idx = cuda.blockIdx.x * cuda.blockDim.x + tid

    # Load element, square it; pad with 0 if out of range
    if idx < n:
        val = data[idx]
        sdata[tid] = val * val
    else:
        sdata[tid] = 0.0

    cuda.syncthreads()

    # Binary tree reduction inside the block
    stride = cuda.blockDim.x // 2
    while stride > 0:
        if tid < stride:
            sdata[tid] += sdata[tid + stride]
        cuda.syncthreads()
        stride //= 2

    # First thread of the block writes the block's partial sum
    if tid == 0:
        partial_sums[cuda.blockIdx.x] = sdata[0]


@cuda.jit
def normalize_kernel(data, norm, n):
    """Divide every element by the scalar L2 norm."""
    idx = cuda.grid(1)
    if idx < n and norm != 0.0:
        data[idx] /= norm

In [4]:
# 4. Setup grid and block dimensions

threads_per_block = 256                     # power of 2 for reduction
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

# Transfer data to device
d_vector = cuda.to_device(host_vector)
d_partial_sums = cuda.device_array(blocks_per_grid, dtype=np.float32)

In [5]:
# 5. Compute L2 norm (reduction)

start = time.time()

# Launch reduction kernel
square_reduce_kernel[blocks_per_grid, threads_per_block](d_vector, d_partial_sums, N)

# Bring partial sums back to CPU and sum them
h_partial_sums = d_partial_sums.copy_to_host()
sum_of_squares = np.sum(h_partial_sums)
l2_norm = math.sqrt(sum_of_squares)   # this is the original norm

In [6]:
# 6. Normalize the whole vector

normalize_kernel[blocks_per_grid, threads_per_block](d_vector, l2_norm, N)

# Copy normalized data back to host
host_normalized = d_vector.copy_to_host()
gpu_time = time.time() - start

In [7]:
# 7. Check the result

norm_after = np.linalg.norm(host_normalized)

print(f"\nOriginal L2 norm : {l2_norm:.6f}")
print(f"Normalized L2 norm : {norm_after:.6f}   (should be 1.0)")
print(f"GPU time : {gpu_time:.4f} seconds")
print("First 5 normalized values:", host_normalized[:5])


Original L2 norm : 1825.720885
Normalized L2 norm : 0.999990   (should be 1.0)
GPU time : 2.9783 seconds
First 5 normalized values: [2.0514643e-04 5.2073365e-04 4.0093419e-04 3.2790253e-04 8.5455911e-05]
